# RoadAnomaly — Phase 4 training (E2a → E2b)

Runs on **Colab GPU**. Artefacts go to Google Drive under `RoadAnomaly/`.

- Set `GITHUB_COMMIT_SHA` to the Stage 2 commit after push.
- Set **both** authorization flags to `True` only for an approved training run.
- Does **not** access the test set.
- Does **not** overwrite Phase 3 artefacts.

In [ ]:
from pathlib import Path
import os

DRIVE_ROOT = Path("/content/drive/MyDrive/RoadAnomaly")
CLONE_DIR = Path("/content/RoadAnomaly")
GITHUB_REPO = "https://github.com/AhmedChatti/RoadAnomaly.git"
GITHUB_BRANCH = "main"
GITHUB_COMMIT_SHA = None  # set to full SHA after Stage 2 push
GITHUB_TOKEN_SECRET_NAME = "GITHUB_TOKEN"

AUTHORIZE_TRAINING = False
AUTHORIZE_IMAGENET_DOWNLOAD = False

EXPECTED_SPLIT = "9a01c05cfc7d33783c4f64b91b0ae423a4d3ad57439fd71ffc029a8f04585c78"
EXPECTED_INFRA = "337cad06211cc09bb32d80fbb897cde15d21f777"

In [ ]:
import subprocess
import sys
from google.colab import drive, userdata

drive.mount("/content/drive")
frames_src = DRIVE_ROOT / "dataset" / "original" / "frames"
assert frames_src.is_dir(), frames_src
print("n_webp", len(list(frames_src.glob("*.webp"))))

os.environ["ROADANOMALY_ARTIFACTS_ROOT"] = str(DRIVE_ROOT)
os.environ["ROADANOMALY_FRAMES_DIR"] = str(frames_src)
os.environ["ROADANOMALY_CHECKPOINTS_DIR"] = str(DRIVE_ROOT / "checkpoints")
os.environ["ROADANOMALY_METADATA_DIR"] = str(DRIVE_ROOT / "experiments" / "metadata")
os.environ["ROADANOMALY_RESULTS_DIR"] = str(DRIVE_ROOT / "experiments" / "results")
os.environ["ROADANOMALY_FIGURES_DIR"] = str(DRIVE_ROOT / "experiments" / "figures")

token = userdata.get(GITHUB_TOKEN_SECRET_NAME)
auth_url = GITHUB_REPO.replace("https://", f"https://x-access-token:{token}@")

def run(cmd, cwd=None):
    print("+", " ".join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)

if CLONE_DIR.exists():
    run(["git", "remote", "set-url", "origin", auth_url], cwd=str(CLONE_DIR))
    run(["git", "fetch", "origin"], cwd=str(CLONE_DIR))
    run(["git", "checkout", GITHUB_BRANCH], cwd=str(CLONE_DIR))
    run(["git", "pull", "origin", GITHUB_BRANCH], cwd=str(CLONE_DIR))
else:
    run(["git", "clone", "--branch", GITHUB_BRANCH, auth_url, str(CLONE_DIR)])

if GITHUB_COMMIT_SHA:
    run(["git", "checkout", GITHUB_COMMIT_SHA], cwd=str(CLONE_DIR))

sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=str(CLONE_DIR), text=True).strip()
print("HEAD", sha)
assert subprocess.run(["git", "merge-base", "--is-ancestor", EXPECTED_INFRA, "HEAD"], cwd=str(CLONE_DIR)).returncode == 0

os.environ["ROADANOMALY_ROOT"] = str(CLONE_DIR)
sys.path.insert(0, str(CLONE_DIR / "src"))
os.chdir(CLONE_DIR)

In [ ]:
import tensorflow as tf
from dataset import load_split_manifest, verify_split_lock

print("TF", tf.__version__)
print("GPU", tf.config.list_physical_devices("GPU"))
split_sha = verify_split_lock(load_split_manifest())
assert split_sha == EXPECTED_SPLIT
print("split_ok", split_sha)

In [ ]:
assert AUTHORIZE_TRAINING and AUTHORIZE_IMAGENET_DOWNLOAD, (
    "Set both AUTHORIZE_TRAINING and AUTHORIZE_IMAGENET_DOWNLOAD to True for training."
)

from phase4_training import train_e2a, train_e2b

e2a = train_e2a(
    authorize_training=True,
    authorize_imagenet_download=True,
    weights="imagenet",
)
print("E2a", e2a["checkpoint_path"], e2a["metadata"]["threshold_selection"]["value"])

e2b = train_e2b(
    e2a_checkpoint=e2a["checkpoint_path"],
    authorize_training=True,
    authorize_imagenet_download=True,
)
print("E2b", e2b["checkpoint_path"], e2b["metadata"]["threshold_selection"]["value"])
print("DONE — test set unused.")